In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

DB_PATH = "/Users/somitagarwal/Desktop/MCA Project/amazon_products.db"
engine = create_engine(f"sqlite:///{DB_PATH}")


def check_and_fix_dtypes(df):
    """
    Har column ka data type check karo aur agar galat hai to fix karo.
    Expected types:
        title        → object (string)
        price        → float64
        rating       → float64
        reviews      → int64
        discount     → float64
        brand        → object (string)
        product_url  → object (string)
    """
    print("\n🔍 Data Types Check Ho Rahi Hai...")
    print("-" * 50)

    expected_types = {
        'title'      : 'object',
        'price'      : 'float64',
        'rating'     : 'float64',
        'reviews'    : 'int64',
        'discount'   : 'float64',
        'brand'      : 'object',
        'product_url': 'object'
    }

    for col, expected in expected_types.items():
        current = str(df[col].dtype)

        # ── Already sahi type hai ──
        if current == expected:
            print(f"'{col}' → Already {expected} hai, kuch karne ki zarurat nahi!")
            continue

        # ── Fix karna padega ──
        print(f"'{col}' → Current: {current} | Expected: {expected} | Fix ho raha hai...")

        try:
            if expected == 'float64':
                df[col] = pd.to_numeric(df[col], errors='coerce')

            elif expected == 'int64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

            elif expected == 'object':
                df[col] = df[col].astype(str)

            print(f"'{col}' successfully {expected} mein convert ho gaya!")

        except Exception as e:
            print(f"'{col}' convert nahi hua! Error: {e}")

    print("-" * 50)
    print("Data Type Check Complete!\n")
    return df


def clean_amazon_data(file_path, site_name):
    """Auto clean function with dtype check"""

    print("Data load ho raha hai...")
    df = pd.read_csv(file_path, skiprows=3)
    print(f"Loaded! Shape: {df.shape}")

    # ── Column Validate Karo ──
    required_columns = ['title', 'price', 'rating', 'reviews', 'discount', 'brand', 'product_url']
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        print(f"Error: Yeh columns nahi mile: {missing_cols}")
        return None

    # ── STEP 1: Data Types Check & Fix ──
    df = check_and_fix_dtypes(df)

    # ── STEP 2: Duplicates Hatao ──
    before = df.shape[0]
    df = df.drop_duplicates()
    print(f"Duplicates removed: {before - df.shape[0]} rows")

    # ── STEP 3: Missing Values Handle Karo ──
    df['rating']   = df['rating'].fillna(df['rating'].median())
    df['discount'] = df['discount'].fillna(0)
    df['brand']    = df['brand'].fillna('Unknown')
    print("Missing values handle ho gaye!")

    # ── STEP 4: Brand Clean Karo ──
    df['brand'] = df['brand'].str.replace('Brand: ', '', regex=False).str.strip()
    print("Brand column clean ho gaya!")

    # ── STEP 5: Original Price Calculate Karo ──
    df['original_price'] = np.where(
        df['discount'] == 0,
        df['price'],
        (df['price'] / (1 - df['discount'] / 100))
    ).round(2)
    print("original_price column add ho gaya!")

    # ── STEP 6: Site Column Add Karo ──
    df['site'] = site_name
    print(f"Site column add ho gaya! → '{site_name}'")

    print("\nData Clean Complete!")
    return df


def save_to_database(df):
    """Clean data ko SQL database mein save karo"""

    print("\nDatabase mein save ho raha hai...")

    try:
        existing_df = pd.read_sql("SELECT title, price FROM amazon_products", engine)

        df['temp_key'] = df['title'] + df['price'].astype(str)
        existing_df['temp_key'] = existing_df['title'] + existing_df['price'].astype(str)

        new_records = df[~df['temp_key'].isin(existing_df['temp_key'])]
        new_records = new_records.drop(columns=['temp_key'])
        df = df.drop(columns=['temp_key'])

    except:
        new_records = df

    if new_records.shape[0] == 0:
        print("Koi naya record nahi mila — database already up to date hai!")
        return

    new_records.to_sql(
        name='amazon_products',
        con=engine,
        if_exists='append',
        index=False
    )

    print(f"{new_records.shape[0]} naye records database mein add ho gaye!")

    total = pd.read_sql("SELECT COUNT(*) as total FROM amazon_products", engine)
    print(f"Database mein ab total records: {total['total'][0]}")


def run_pipeline(file_path):
    """CSV → Clean → SQL"""

    print("=" * 50)
    print("Pipeline Start!")
    print("=" * 50)

    # ── User se site ka naam lo ──
    print("\nSupported Sites: Amazon, Flipkart, Meesho, Myntra, etc.")
    site_name = input("Kis site ka data hai? Site ka naam likho: ").strip()

    if site_name == "":
        print("Site naam blank nahi ho sakta!")
        return

    print(f"\nSite set ho gayi: '{site_name}'")
    print("=" * 50)

    # Step 1: Clean
    df = clean_amazon_data(file_path, site_name)

    if df is None:
        print("Pipeline failed — data clean nahi hua")
        return

    # Step 2: Database mein Save
    save_to_database(df)

    print("\nPipeline Complete!")
    print("=" * 50)

run_pipeline("/Users/somitagarwal/Desktop/MCA Project/Uncleaned data/merge-csv_com__6994ddc3d422d.csv")


🚀 Pipeline Start!

📌 Supported Sites: Amazon, Flipkart, Meesho, Myntra, etc.


👉 Kis site ka data hai? Site ka naam likho:  Amazon



✅ Site set ho gayi: 'Amazon'
📂 Data load ho raha hai...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/somitagarwal/Desktop/MCA Project/Uncleaned data/merge-csv_com__6994ddc3d422d.csv'